# p109 late-training reorganization (~27k)

`p109_seed485_dseed598` grokked early, sat stable ~5k–26k, then spiked hard at ~27k and re-settled by ~30k. Three instruments (parameter PCA, activation-DMD eigenvalues, DMD residuals) fire in the same window, but the committed frequencies (4/14/27) survive. This notebook works the event in five passes:

1. **Event localization** — which neurons explode, and how concentrated is it?
2. **Grouping reconciliation** — the clustering artifact vs. the distribution view partition neurons differently. Pin down the rule before counting anything.
3. **Frequency switching** — did neurons change frequency? Gate on confidence to separate real reassignment from clustering jitter.
4. **Freq-8 transient + attention lead** — the sub-dominant swell that complicates 'none born', and the propagation order.
5. **Procrustes: gauge vs functional** — is the upstream (embedding) move a rotation the MLP mostly ignores, leaving the ~11 exploders as the functional residue?

First pass lives in `apps/research/sketches/p109_event_neuron_displacement.py`; this notebook lifts and extends it. All data access goes through the miscope API (`variant.artifacts`), not file paths.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

# Locate the repo root (holds data/) and chdir there so relative data paths resolve
# from any launch directory; then make the first-pass sketch importable.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "modulo_addition_1layer").exists())
os.chdir(root)
sys.path.insert(0, str(root / "apps" / "research" / "sketches"))
import p109_event_neuron_displacement as fp  # load_w_in_trajectory, freq_assignments, windows
from miscope.families.discovery import load_family_from_dir
from miscope.views.universal import _adapt_activation_freq_legacy

fam = load_family_from_dir("data/modulo_addition_1layer", "data")
variant = fam.get_variant(prime=109, seed=485, data_seed=598)
variant

## 1. Event localization — which neurons explode?

Rank neurons by peak `W_in`-column excursion during the event, in units of each neuron's own normal plateau drift. The handful of outliers carry the globally-visible weight-space departure.

In [ ]:
epochs, W = fp.load_w_in_trajectory(variant)          # (E, N, D)
assign = fp.freq_assignments(variant, fp.GROUP_EPOCH)  # row-index per neuron (see section 2)

plateau = fp._window_mask(epochs, *fp.PLATEAU)
event = fp._window_mask(epochs, *fp.EVENT)
post_idx = int(np.argmin(np.abs(epochs - fp.POST)))

ref = W[plateau].mean(axis=0)
normal_step = np.linalg.norm(np.diff(W[plateau], axis=0), axis=2).mean(axis=0) + 1e-9
peak_disp = np.linalg.norm(W[event] - ref[None], axis=2).max(axis=0)
net_disp = np.linalg.norm(W[post_idx] - ref, axis=1)
score = peak_disp / normal_step

order = np.argsort(score)[::-1]
print("top exploders (neuron, freq-row, score×, peak, net):")
for j in order[:15]:
    print(f"  {j:>3}  row={assign[j]:>2}  {score[j]:>8.0f}×  peak={peak_disp[j]:.2f}  net={net_disp[j]:.2f}")

In [ ]:
# Per-epoch excursion traces for the top movers — do they leave and return, or relocate?
fig = go.Figure()
for j in order[:6]:
    disp = np.linalg.norm(W - ref[None], axis=2)[:, j]
    fig.add_trace(go.Scatter(x=epochs, y=disp, mode="lines", name=f"n{j} (row {assign[j]})"))
fig.update_layout(title="Top-6 exploder W_in excursion from plateau reference",
                  xaxis_title="epoch", yaxis_title="||W_in[:,j] - ref||",
                  xaxis_range=[24000, 30000], height=420)
fig.show()

## 2. Grouping reconciliation

Two surfaces partition neurons by frequency and they disagree:

- `neuron_grouping.assignments` clusters all 512 neurons — sizes ~274/95/143.
- The `neuron_freq_distribution` view (committed/dominant rule) — sizes 183/134/195 in the plots.

**Resolved indexing:** `mlp_out_frequencies` maps row index `k → frequency k+1`. So the grouping's `3/13/26` are *row indices* = actual frequencies **4/14/27**, matching the plot labels. The count gap is a genuine membership-rule difference (cluster-assignment vs. dominant-frequency), not an indexing bug. Decide which rule is authoritative for switch-counting below.

In [ ]:
def norm_matrix(epoch):
    art = variant.artifacts.load_epoch("activation_basis_projection", epoch)
    return _adapt_activation_freq_legacy(art, "mlp_out", "norm_matrix")["norm_matrix"]  # (F, N)

freqs = variant.artifacts.load_epoch("activation_basis_projection", fp.GROUP_EPOCH)["mlp_out_frequencies"]
print("row->freq map (first 16):", list(zip(range(16), freqs[:16].tolist())))

# Cluster sizes (row indices) vs dominant-frequency membership at the same epoch.
u, c = np.unique(assign, return_counts=True)
print("\ncluster sizes (row -> count):", {int(k): int(v) for k, v in zip(u, c)})
nm = norm_matrix(fp.GROUP_EPOCH)
dom = np.argmax(nm, axis=0)
u2, c2 = np.unique(dom, return_counts=True)
top = sorted(zip(u2.tolist(), c2.tolist()), key=lambda x: -x[1])[:5]
print("dominant-freq sizes (row -> count, top 5):", {int(freqs[k] - 1): v for k, v in top}, "<- as rows")

## 3. Frequency switching (confidence-gated)

The switch guess, tested. Group sizes do shift directionally (freq-14 grows, freq-27 shrinks), but the switchers are the low-confidence neurons — the committed core holds.

In [ ]:
fp.frequency_switch_report(variant)

## 4. Freq-8 transient + attention lead

Frequency 8 (row index 7) swells ~2× across 28100–28700 then recedes — but never becomes any neuron's dominant frequency. A threshold-sensitive 'none born' caveat. Separately, attention-out's DMD residual climbs before MLP-out's spike; freq-8 also shows up as a dominant *pair* in the attention spectra.

In [ ]:
i8 = int(np.where(freqs == 8)[0][0])  # actual frequency 8 -> row 7
win = [e for e in epochs if 26500 <= e <= 30000]
power8, dom8 = [], []
for e in win:
    m = norm_matrix(int(e))
    power8.append(m[i8].sum())
    d = np.argmax(m, axis=0)
    df = m[d, np.arange(m.shape[1])]
    dom8.append(int(((d == i8) & (df >= 0.10)).sum()))

fig = go.Figure()
fig.add_trace(go.Scatter(x=win, y=power8, mode="lines+markers", name="freq-8 total power"))
fig.add_trace(go.Scatter(x=win, y=dom8, mode="lines+markers", name="neurons dominant@10%", yaxis="y2"))
fig.update_layout(title="Freq-8: transient power swell, zero dominance",
                  xaxis_title="epoch", yaxis_title="total power",
                  yaxis2=dict(title="dominant-neuron count", overlaying="y", side="right"),
                  height=420)
fig.show()

### Open threads

- Do the ~11 exploders leave-and-return or genuinely relocate? (peak vs net already disagree)
- Settle the authoritative grouping rule (section 2) before any published switch count.
- Are the exploded neurons the ones attention was routing to when its residual started climbing (~23–24k)?
- Why does loosened capacity re-commit preferentially to freq-14?
- **Fold the attention path into the Procrustes test (§5).** The embedding move is ~86% a `d_model` rotation, but co-rotating `W_in` by that same `R` makes the fit *worse* (negative gauge fraction) — because attention sits between embedding and MLP, so the residual stream the MLP reads is not the raw rotated embedding. The clean gauge test is: recover `R` from `W_E`, verify `W_Q/W_K/W_V` co-rotate by `Rᵀ` on their `d_model` axis, then measure the MLP residual against the *attention-output* frame, not the embedding frame. (5/11 exploders still top the embedding-frame residual, incl. the extreme n327 — partial signal through the frame mismatch.)

In [ ]:
from scipy.linalg import orthogonal_procrustes


def gauge_split(A, B):
    """Best orthogonal R with A@R ~ B; how much of the A->B move is rotational."""
    R, _ = orthogonal_procrustes(A, B)
    raw = np.linalg.norm(B - A)
    resid = np.linalg.norm(B - A @ R)
    gauge_frac = 1 - (resid / raw) ** 2 if raw > 0 else float("nan")
    return R, raw, resid, gauge_frac


PRE, POST = 26000, int(epochs[post_idx])
we = lambda e: variant.artifacts.load_epoch("parameter_snapshot", e)["W_E"].astype(np.float64)

R, raw, resid, gf = gauge_split(we(PRE), we(POST))       # embedding move across the event
R0, raw0, resid0, gf0 = gauge_split(we(24000), we(26000))  # plateau null for calibration

print("W_E embedding move — d_model rotation via orthogonal Procrustes (W_E_pre @ R ~ W_E_post)")
print(f"  event {PRE}->{POST}: ||ΔW_E||={raw:7.3f}  aligned resid={resid:7.3f}  gauge-explained={gf:6.1%}")
print(f"  null  24000->26000 : ||ΔW_E||={raw0:7.3f}  aligned resid={resid0:7.3f}  gauge-explained={gf0:6.1%}")
print(f"  rotation size ||R-I||_F (event) = {np.linalg.norm(R - np.eye(R.shape[0])):.3f}")

In [ ]:
# If the embedding basis rotates by R (resid' = resid @ R), the gauge-consistent MLP
# co-rotation is W_in' = Rᵀ @ W_in (keeps hidden = resid @ W_in invariant). The
# residual after co-rotation is the part of the MLP move NOT explained by the
# embedding rotation — the functional component. Test: is it carried by the
# section-1 exploders?
pre_idx = int(np.argmin(np.abs(epochs - PRE)))
W_in_pre, W_in_post = W[pre_idx].T, W[post_idx].T          # (d_model, n_neurons)
pred = R.T @ W_in_pre                                       # gauge prediction
raw_col = np.linalg.norm(W_in_post - W_in_pre, axis=0)      # per-neuron raw move
resid_col = np.linalg.norm(W_in_post - pred, axis=0)        # per-neuron co-rotation residual
mlp_gauge_frac = 1 - (np.linalg.norm(W_in_post - pred) / np.linalg.norm(W_in_post - W_in_pre)) ** 2
print(f"MLP move gauge-explained by the embedding rotation: {mlp_gauge_frac:.1%}")

exploders = list(order[:11])
resid_rank = np.argsort(resid_col)[::-1].tolist()
overlap = len(set(resid_rank[:11]) & set(exploders))
print(f"top-11 by co-rotation residual: {resid_rank[:11]}")
print(f"section-1 exploders          : {exploders}")
print(f"overlap: {overlap}/11  (high overlap => exploders ARE the functional residue)")

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.arange(len(resid_col)), y=resid_col, mode="markers",
                         name="all neurons", marker=dict(size=4, color="lightgray")))
fig.add_trace(go.Scatter(x=exploders, y=resid_col[exploders], mode="markers",
                         name="section-1 exploders", marker=dict(size=10, color="crimson")))
fig.update_layout(title="Per-neuron MLP residual after embedding-rotation co-rotation",
                  xaxis_title="neuron", yaxis_title="||W_in_post - Rᵀ·W_in_pre||", height=420)
fig.show()

### Open threads

- Do the ~11 exploders leave-and-return or genuinely relocate? (peak vs net already disagree)
- Settle the authoritative grouping rule (section 2) before any published switch count.
- Are the exploded neurons the ones attention was routing to when its residual started climbing (~23–24k)?
- Why does loosened capacity re-commit preferentially to freq-14?